In [ ]:

!pip install pandas nltk scikit-learn kagglehub

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn import metrics
import kagglehub


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hsankesara/medium-articles")

print("Path to dataset files:", path)
import pandas as pd
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("hsankesara/medium-articles")
# Find the CSV file within the downloaded dataset
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_file_path = os.path.join(root, file)
            break  # Stop searching once the CSV file is found
    else:
        continue  # Continue searching in other subdirectories
    break  # Stop searching once the CSV file is found in a subdirectory

# Load the CSV file
data = pd.read_csv(csv_file_path)

Path to dataset files: /root/.cache/kagglehub/datasets/hsankesara/medium-articles/versions/1


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 337 entries, 0 to 336
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   author        337 non-null    object
 1   claps         337 non-null    object
 2   reading_time  337 non-null    int64 
 3   link          337 non-null    object
 4   title         337 non-null    object
 5   text          337 non-null    object
dtypes: int64(1), object(5)
memory usage: 15.9+ KB


In [ ]:
data.head()

,author,claps,reading_time,link,title,text
0,Justin Lee,8.3K,11,https://medium.com/swlh/chatbots-were-the-next...,Chatbots were the next big thing: what happene...,"Oh, how the headlines blared:\nChatbots were T..."
1,Conor Dewey,1.4K,7,https://towardsdatascience.com/python-for-data...,Python for Data Science: 8 Concepts You May Ha...,If you’ve ever found yourself looking up the s...
2,William Koehrsen,2.8K,11,https://towardsdatascience.com/automated-featu...,Automated Feature Engineering in Python – Towa...,Machine learning is increasingly moving from h...
3,Gant Laborde,1.3K,7,https://medium.freecodecamp.org/machine-learni...,Machine Learning: how to go from Zero to Hero ...,If your understanding of A.I. and Machine Lear...
4,Emmanuel Ameisen,935,11,https://blog.insightdatascience.com/reinforcem...,Reinforcement Learning from scratch – Insight ...,Want to learn about applied Artificial Intelli...


In [ ]:
print(data.columns)

Index(['author', 'claps', 'reading_time', 'link', 'title', 'text'], dtype='object')


In [ ]:
data.drop(['author', 'claps', 'reading_time', 'link'], axis=1, inplace=True)

In [ ]:
data.head()

,title,text,cleaned_text,cleaned_title
0,Chatbots were the next big thing: what happene...,"Oh, how the headlines blared:\nChatbots were T...",oh headline blared chatbots next big thing hop...,chatbots next big thing happened startup medium
1,Python for Data Science: 8 Concepts You May Ha...,If you’ve ever found yourself looking up the s...,youve ever found looking question concept synt...,python data science concept may forgotten
2,Automated Feature Engineering in Python – Towa...,Machine learning is increasingly moving from h...,machine learning increasingly moving handdesig...,automated feature engineering python towards d...
3,Machine Learning: how to go from Zero to Hero ...,If your understanding of A.I. and Machine Lear...,understanding ai machine learning big question...,machine learning go zero hero freecodecamp
4,Reinforcement Learning from scratch – Insight ...,Want to learn about applied Artificial Intelli...,want learn applied artificial intelligence lea...,reinforcement learning scratch insight data


In [ ]:
# تحميل بيانات nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')
# دالة لتنظيف النصوص
def preprocess_text(text):
    # تحويل النص إلى lowercase
    text = text.lower()
    # إزالة الأرقام
    text = re.sub(r'\d+', '', text)
    # إزالة علامات الترقيم
    text = re.sub(r'[^\w\s]', '', text)
    # إزالة الكلمات الشائعة (stopwords)
    stop_words = set(stopwords.words('english'))
    words = nltk.word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    # استخدام lemmatization
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

# تطبيق المعالجة المسبقة على عمود العناوين
data['cleaned_text'] = data['text'].apply(preprocess_text)
data['cleaned_title']=data['title'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
data.head()

,title,text,cleaned_text,cleaned_title
0,Chatbots were the next big thing: what happene...,"Oh, how the headlines blared:\nChatbots were T...",oh headline blared chatbots next big thing hop...,chatbots next big thing happened startup medium
1,Python for Data Science: 8 Concepts You May Ha...,If you’ve ever found yourself looking up the s...,youve ever found looking question concept synt...,python data science concept may forgotten
2,Automated Feature Engineering in Python – Towa...,Machine learning is increasingly moving from h...,machine learning increasingly moving handdesig...,automated feature engineering python towards d...
3,Machine Learning: how to go from Zero to Hero ...,If your understanding of A.I. and Machine Lear...,understanding ai machine learning big question...,machine learning go zero hero freecodecamp
4,Reinforcement Learning from scratch – Insight ...,Want to learn about applied Artificial Intelli...,want learn applied artificial intelligence lea...,reinforcement learning scratch insight data


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# تحويل النصوص إلى تمثيلات عددية باستخدام CountVectorizer (يمكنك تعديل max_features حسب الحاجة)
vectorizer = CountVectorizer(max_features=5000)
X = vectorizer.fit_transform(data['cleaned_text']).toarray()  # تحويل النصوص إلى مصفوفة عددية


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# تقسيم البيانات إلى مجموعة تدريب واختبار
X_train, X_test, y_train, y_test = train_test_split(X, data['cleaned_title'], test_size=0.4, random_state=50)

# تدريب نموذج RandomForest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# التنبؤ بالعناوين
y_pred = model.predict(X_test)

# حساب الدقة
accuracy = accuracy_score(y_test, y_pred)
print(f"accuracy: {accuracy * 100:.2f}%")


accuracy: 43.70%


In [ ]:
# التنبؤ بالعناوين بناءً على مجموعة الاختبار
y_pred = model.predict(X_test)

# عرض أول 5 نتائج للتنبؤ
print(" (Predictions):")
for i in range(5):
    print(f"text: {X_test[i]}")
    print(f"pred_title: {y_pred[i]}")
    print("-" * 50)

 (Predictions):
text: [0 0 0 ... 0 1 1]
pred_title: machine learning fun part adam geitgey medium
--------------------------------------------------
text: [0 0 1 ... 0 0 0]
pred_title: cheat sheet ai neural network machine learning deep learning big data
--------------------------------------------------
text: [0 0 0 ... 0 0 0]
pred_title: digital companionship matt wiese medium
--------------------------------------------------
text: [0 0 3 ... 0 0 0]
pred_title: every single machine learning course internet ranked review
--------------------------------------------------
text: [0 0 1 ... 0 0 0]
pred_title: machine learning เรยนอะไร รไปทาไม v e r f e medium
--------------------------------------------------
